# Semana 3: Regresión Lineal y Regularización
## Notebook de Ejercicios (NB2) – Predicción de Precios de Viviendas

**Propósito:** Aplicar los conceptos de regresión lineal y regularización sobre un dataset real para predecir el valor medio de las viviendas en California.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Dividir correctamente los datos en entrenamiento y prueba.
- Entrenar una regresión lineal y evaluar su rendimiento con RMSE, MAE y R².
- Aplicar Ridge, Lasso y ElasticNet con validación cruzada (GridSearchCV).
- Analizar los coeficientes y la selección de características (especialmente en Lasso).
- Comparar el rendimiento de los modelos regularizados vs el modelo lineal simple.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Carga del Dataset: California Housing

El dataset **California Housing** está disponible directamente en scikit-learn. Contiene información sobre viviendas en distritos de California con el objetivo de predecir el valor mediano de las casas (expresado en cientos de miles de dólares, aproximadamente).

**Características:**
- `MedInc`: Ingreso mediano en el bloque (en decenas de miles de dólares).
- `HouseAge`: Edad mediana de las casas en el bloque.
- `AveRooms`: Número promedio de habitaciones por vivienda.
- `AveBedrms`: Número promedio de dormitorios por vivienda.
- `Population`: Población del bloque.
- `AveOccup`: Promedio de ocupantes por vivienda.
- `Latitude`: Latitud.
- `Longitude`: Longitud.

**Variable objetivo:** `MedHouseVal` – Valor mediano de las casas (en cientos de miles de dólares).

In [ ]:
# Cargamos el dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(f"Dimensiones del dataset: {df.shape}")
print("\nPrimeras 5 filas:")
df.head()

In [ ]:
# Información general
df.info()

In [ ]:
# Estadísticas descriptivas
df.describe()

### 1.1. Breve Análisis Exploratorio

Visualizamos rápidamente la distribución de la variable objetivo y las correlaciones.

In [ ]:
# Distribución de la variable objetivo
plt.figure(figsize=(8, 4))
sns.histplot(df['MedHouseVal'], kde=True, bins=50)
plt.title('Distribución del Valor Medio de las Viviendas')
plt.xlabel('MedHouseVal (cientos de miles de $)')
plt.show()

In [ ]:
# Matriz de correlación
plt.figure(figsize=(10, 8))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Matriz de Correlación')
plt.show()

---
## 2. División en Train/Test y Preprocesamiento

Separamos las características (X) de la variable objetivo (y). Dividimos en entrenamiento (80%) y prueba (20%).

**Importante:** Escalamos las variables numéricas para que la regularización funcione correctamente.

In [ ]:
# Definimos X e y
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

# Dividimos en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamaño de entrenamiento: {X_train.shape}")
print(f"Tamaño de prueba: {X_test.shape}")

In [ ]:
# Escalamos las características (importante para regularización)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertimos a DataFrame para mantener nombres
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Escalado completado.")
X_train_scaled.describe()

---
## 3. Regresión Lineal Simple (sin regularización)

Entrenamos un modelo de regresión lineal estándar y evaluamos su rendimiento.

In [ ]:
# Entrenamos modelo
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# Predicciones
y_pred_lr = lr.predict(X_test_scaled)

# Métricas
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("=== Regresión Lineal Simple ===")
print(f"RMSE: {rmse_lr:.4f}")
print(f"MAE: {mae_lr:.4f}")
print(f"R²: {r2_lr:.4f}")

# Coeficientes
coef_df = pd.DataFrame({
    'Característica': X.columns,
    'Coeficiente': lr.coef_
})
print("\nCoeficientes del modelo:")
print(coef_df.sort_values('Coeficiente', ascending=False))

---
## 4. Ridge Regression (L2) con Validación Cruzada

Usamos `GridSearchCV` para encontrar el mejor hiperparámetro alpha.

In [ ]:
# Definimos el rango de alphas
alphas = {'alpha': np.logspace(-3, 3, 50)}

# Configuramos GridSearchCV
ridge = Ridge()
ridge_cv = GridSearchCV(ridge, alphas, cv=5, scoring='neg_mean_squared_error', verbose=1)
ridge_cv.fit(X_train_scaled, y_train)

print(f"Mejor alpha para Ridge: {ridge_cv.best_params_['alpha']:.4f}")
print(f"Mejor MSE (negativo): {-ridge_cv.best_score_:.4f}")

# Evaluamos en test
y_pred_ridge = ridge_cv.predict(X_test_scaled)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

print("\n=== Ridge (mejor modelo) ===")
print(f"RMSE: {rmse_ridge:.4f}")
print(f"MAE: {mae_ridge:.4f}")
print(f"R²: {r2_ridge:.4f}")

In [ ]:
# Coeficientes del mejor modelo Ridge
coef_ridge = ridge_cv.best_estimator_.coef_
coef_ridge_df = pd.DataFrame({
    'Característica': X.columns,
    'Coeficiente': coef_ridge
})
print("Coeficientes del mejor modelo Ridge:")
print(coef_ridge_df.sort_values('Coeficiente', ascending=False))

---
## 5. Lasso Regression (L1) con Validación Cruzada

Lasso puede llevar algunos coeficientes exactamente a cero, realizando selección de características.

In [ ]:
# Lasso puede necesitar más iteraciones
lasso = Lasso(max_iter=10000)
lasso_cv = GridSearchCV(lasso, alphas, cv=5, scoring='neg_mean_squared_error', verbose=1)
lasso_cv.fit(X_train_scaled, y_train)

print(f"Mejor alpha para Lasso: {lasso_cv.best_params_['alpha']:.4f}")
print(f"Mejor MSE (negativo): {-lasso_cv.best_score_:.4f}")

# Evaluamos en test
y_pred_lasso = lasso_cv.predict(X_test_scaled)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
r2_lasso = r2_score(y_test, y_pred_lasso)

print("\n=== Lasso (mejor modelo) ===")
print(f"RMSE: {rmse_lasso:.4f}")
print(f"MAE: {mae_lasso:.4f}")
print(f"R²: {r2_lasso:.4f}")

In [ ]:
# Coeficientes del mejor modelo Lasso
coef_lasso = lasso_cv.best_estimator_.coef_
coef_lasso_df = pd.DataFrame({
    'Característica': X.columns,
    'Coeficiente': coef_lasso
})
print("Coeficientes del mejor modelo Lasso:")
print(coef_lasso_df.sort_values('Coeficiente', ascending=False))

# Identificamos variables seleccionadas (coeficientes distintos de cero)
selected_features = coef_lasso_df[coef_lasso_df['Coeficiente'] != 0]['Característica'].tolist()
print(f"\nLasso ha seleccionado {len(selected_features)} de {len(X.columns)} características:")
print(selected_features)

---
## 6. Elastic Net con Validación Cruzada

Elastic Net combina las penalizaciones L1 y L2. Necesitamos buscar tanto alpha como el ratio de mezcla (l1_ratio).

In [ ]:
# Definimos el grid de hiperparámetros
param_grid = {
    'alpha': np.logspace(-3, 2, 20),
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1]
}

elastic = ElasticNet(max_iter=10000)
elastic_cv = GridSearchCV(elastic, param_grid, cv=5, scoring='neg_mean_squared_error', verbose=1, n_jobs=-1)
elastic_cv.fit(X_train_scaled, y_train)

print(f"Mejores parámetros para Elastic Net:")
print(f"alpha: {elastic_cv.best_params_['alpha']:.4f}")
print(f"l1_ratio: {elastic_cv.best_params_['l1_ratio']}")
print(f"Mejor MSE (negativo): {-elastic_cv.best_score_:.4f}")

# Evaluamos en test
y_pred_elastic = elastic_cv.predict(X_test_scaled)
rmse_elastic = np.sqrt(mean_squared_error(y_test, y_pred_elastic))
mae_elastic = mean_absolute_error(y_test, y_pred_elastic)
r2_elastic = r2_score(y_test, y_pred_elastic)

print("\n=== Elastic Net (mejor modelo) ===")
print(f"RMSE: {rmse_elastic:.4f}")
print(f"MAE: {mae_elastic:.4f}")
print(f"R²: {r2_elastic:.4f}")

In [ ]:
# Coeficientes del mejor modelo Elastic Net
coef_elastic = elastic_cv.best_estimator_.coef_
coef_elastic_df = pd.DataFrame({
    'Característica': X.columns,
    'Coeficiente': coef_elastic
})
print("Coeficientes del mejor modelo Elastic Net:")
print(coef_elastic_df.sort_values('Coeficiente', ascending=False))

# Identificamos variables seleccionadas
selected_elastic = coef_elastic_df[coef_elastic_df['Coeficiente'] != 0]['Característica'].tolist()
print(f"\nElastic Net ha seleccionado {len(selected_elastic)} de {len(X.columns)} características:")
print(selected_elastic)

---
## 7. Comparación de Modelos

Creamos una tabla resumen con las métricas de todos los modelos para comparar su rendimiento.

In [ ]:
# Creamos DataFrame de comparación
comparacion = pd.DataFrame({
    'Modelo': ['Regresión Lineal', 'Ridge', 'Lasso', 'Elastic Net'],
    'RMSE': [rmse_lr, rmse_ridge, rmse_lasso, rmse_elastic],
    'MAE': [mae_lr, mae_ridge, mae_lasso, mae_elastic],
    'R²': [r2_lr, r2_ridge, r2_lasso, r2_elastic]
})

print("=== Comparación de Modelos ===")
comparacion.round(4)

### 7.1. Visualización de la comparación

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# RMSE
sns.barplot(data=comparacion, x='Modelo', y='RMSE', ax=axes[0])
axes[0].set_title('RMSE por Modelo')
axes[0].tick_params(axis='x', rotation=45)

# MAE
sns.barplot(data=comparacion, x='Modelo', y='MAE', ax=axes[1])
axes[1].set_title('MAE por Modelo')
axes[1].tick_params(axis='x', rotation=45)

# R²
sns.barplot(data=comparacion, x='Modelo', y='R²', ax=axes[2])
axes[2].set_title('R² por Modelo')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 8. Análisis de Coeficientes y Selección de Características

Comparamos los coeficientes de los diferentes modelos para entender el efecto de la regularización.

In [ ]:
# Creamos un DataFrame con todos los coeficientes
coef_comparison = pd.DataFrame({
    'Característica': X.columns,
    'Linear': lr.coef_,
    'Ridge': coef_ridge,
    'Lasso': coef_lasso,
    'Elastic': coef_elastic
})

print("Comparación de coeficientes:")
coef_comparison.round(4)

In [ ]:
# Visualizamos los coeficientes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, (modelo, color) in enumerate(zip(['Linear', 'Ridge', 'Lasso', 'Elastic'], ['blue', 'green', 'red', 'purple'])):
    axes[i].barh(coef_comparison['Característica'], coef_comparison[modelo], color=color)
    axes[i].set_title(f'Coeficientes - {modelo}')
    axes[i].axvline(x=0, color='black', linestyle='--', linewidth=0.5)
    axes[i].grid(axis='x')

plt.tight_layout()
plt.show()

### 8.1. Discusión sobre la selección de características

**Observaciones:**

- **Regresión Lineal:** Todos los coeficientes son distintos de cero. Algunos pueden ser grandes pero inestables.
- **Ridge:** Todos los coeficientes se encogen hacia cero, pero ninguno es exactamente cero. Esto es útil cuando todas las variables aportan información.
- **Lasso:** Ha seleccionado solo un subconjunto de variables. Las variables con coeficiente cero son:
  ```python
  [feature for feature, coef in zip(X.columns, coef_lasso) if coef == 0]
  ```
- **Elastic Net:** Similar a Lasso pero con un comportamiento más estable cuando hay variables correlacionadas.

**Preguntas para reflexionar:**
1. ¿Qué variables parecen más importantes según los coeficientes?
2. ¿Por qué Lasso elimina algunas variables mientras que Ridge no?
3. En términos de negocio, ¿qué ventajas tiene un modelo más simple (con menos variables)?

In [ ]:
# Mostramos las variables eliminadas por Lasso
zero_coef_lasso = coef_comparison[coef_comparison['Lasso'] == 0]['Característica'].tolist()
print(f"Variables eliminadas por Lasso (coeficiente = 0): {zero_coef_lasso}")

# Si no hay ninguna, mostramos un mensaje
if not zero_coef_lasso:
    print("Nota: Con el alpha seleccionado, Lasso no eliminó ninguna variable.")
    print("Prueba a aumentar alpha para ver selección.")

---
## 9. Conclusiones

En este notebook hemos aplicado técnicas de regresión y regularización a un problema real de predicción de precios de viviendas:

✔️ **Regresión Lineal:** Modelo base, interpretable pero propenso a sobreajuste si hay muchas variables.
✔️ **Ridge (L2):** Mejora la estabilidad reduciendo la magnitud de los coeficientes. Útil cuando hay multicolinealidad.
✔️ **Lasso (L1):** Realiza selección de variables, produciendo modelos más simples y a menudo más interpretables.
✔️ **Elastic Net:** Combina lo mejor de ambos, especialmente útil con variables correlacionadas.
✔️ **Validación Cruzada:** Fundamental para elegir los hiperparámetros de regularización.

**Lección clave:** La regularización no siempre mejora drásticamente el RMSE, pero sí proporciona modelos más robustos y, en el caso de Lasso, más interpretables al seleccionar las características más relevantes.

En la siguiente sesión aplicaremos estos conceptos a problemas de clasificación con regresión logística.

---
**Fin del Notebook de Ejercicios - Semana 3**